In [4]:
from pybaseball import statcast, playerid_reverse_lookup
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, brier_score_loss, log_loss
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.calibration import calibration_curve
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pickle

/Users/briandeng/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
def get_statcast_range_safe(start_date, end_date, freq='30D'):
    all_data = []
    current_start = pd.to_datetime(start_date)
    final_date = pd.to_datetime(end_date)

    while current_start <= final_date:
        current_end = min(current_start + pd.Timedelta(freq), final_date)
        print(f"Fetching {current_start.date()} to {current_end.date()}...")
        try:
            data_chunk = statcast(start_dt=str(current_start.date()), end_dt=str(current_end.date()))
            all_data.append(data_chunk)
        except Exception as e:
            print(f"Failed: {e}")
        current_start = current_end + timedelta(days=1)
    
    return pd.concat(all_data, ignore_index=True)

In [6]:
# select 2021 - 2023 regular season data
data_2021 = get_statcast_range_safe("2021-04-01", "2021-10-03")
data_2022 = get_statcast_range_safe("2022-04-07", "2022-10-05")
data_2023 = get_statcast_range_safe("2023-03-30", "2023-10-01")

Fetching 2021-04-01 to 2021-05-01...
This is a large query, it may take a moment to complete


  0%|          | 0/31 [00:00<?, ?it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  3%|▎         | 1/31 [00:00<00:15,  1.92it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  6%|▋         | 2/31 [00:00<00:08,  3.61it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore'

Fetching 2021-05-02 to 2021-06-01...
This is a large query, it may take a moment to complete


  0%|          | 0/31 [00:00<?, ?it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  3%|▎         | 1/31 [00:00<00:17,  1.68it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  6%|▋         | 2/31 [00:00<00:09,  3.00it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore'

Fetching 2021-06-02 to 2021-07-02...
This is a large query, it may take a moment to complete


...

  3%|▎         | 1/31 [00:00<00:04,  6.92it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
 16%|█▌        | 5/31 [00:00<00:04,  5.27it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
 19%|█▉        | 6/31 [00:01<00:04,  5.21it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors=

Fetching 2023-08-01 to 2023-08-31...
This is a large query, it may take a moment to complete


  0%|          | 0/31 [00:00<?, ?it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  3%|▎         | 1/31 [00:00<00:28,  1.07it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  6%|▋         | 2/31 [00:01<00:13,  2.19it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore'

Fetching 2023-09-01 to 2023-10-01...
This is a large query, it may take a moment to complete


  0%|          | 0/31 [00:00<?, ?it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  3%|▎         | 1/31 [00:00<00:20,  1.48it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
  6%|▋         | 2/31 [00:00<00:09,  2.96it/s]/Users/briandeng/Library/Python/3.9/lib/python/site-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore'

In [7]:
# dombine data frame
all_data = pd.concat([data_2021, data_2022, data_2023], ignore_index=True)

In [8]:
all_data.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,CU,2021-05-01,83.0,-1.14,5.89,"White, Mitch",543768,669952,single,hit_into_play,...,2,4.21,-0.24,0.24,49.3,<NA>,<NA>,<NA>,<NA>,<NA>
1,FF,2021-05-01,94.0,-1.13,5.98,"White, Mitch",543768,669952,NaN,foul,...,2,1.06,0.23,-0.23,51.3,<NA>,<NA>,<NA>,<NA>,<NA>
2,CU,2021-05-01,82.4,-1.12,5.96,"White, Mitch",543768,669952,NaN,called_strike,...,2,4.47,-0.3,0.3,50.6,<NA>,<NA>,<NA>,<NA>,<NA>
3,FF,2021-05-01,95.5,-1.0,5.94,"White, Mitch",541645,669952,single,hit_into_play,...,1,1.06,0.48,0.48,51.4,<NA>,<NA>,<NA>,<NA>,<NA>
4,SL,2021-05-01,88.8,-1.14,5.8,"White, Mitch",541645,669952,NaN,blocked_ball,...,1,2.36,-0.11,-0.11,48.7,<NA>,<NA>,<NA>,<NA>,<NA>


In [9]:
# create pitch type label dictionary
PITCH_TYPE = {}
pitch_type = all_data["pitch_type"].unique()
for i in range(len(pitch_type)):
    PITCH_TYPE[i] = pitch_type[i]
    PITCH_TYPE[pitch_type[i]] = i

In [10]:
# save pitch type dictionary
with open("pitch_type.pkl", "wb") as f:
    pickle.dump(PITCH_TYPE, f)

In [11]:
# create pitch category
PITCH_CAT = {
    'fastball': ['FF', 'SI', 'FC', 'FA'],
    'breaking': ['SL', 'CU', 'ST', 'SV', 'KC', 'SC', 'KN'],
    'offspeed': ['CH', 'FS', 'FO', 'CS', 'EP']
}

In [12]:
# search batter_name function
def search_batter_name(batter):
    """
    Return given batter name.
    """

    batter_info = playerid_reverse_lookup([batter])
    batter_info['batter_name'] = batter_info['name_first'] + ' ' + batter_info['name_last']
    
    return batter_info['batter_name'][0]

# apply to dataframe
batters = all_data["batter"].unique()
batter_dict = {} # batter name dictionary
for batter in batters:
    batter_dict[batter] = search_batter_name(batter)
all_data["batter_name"] = all_data["batter"].map(batter_dict)

Gathering player lookup table. This may take a moment.


In [13]:
all_data.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,batter_name
0,CU,2021-05-01,83.0,-1.14,5.89,"White, Mitch",543768,669952,single,hit_into_play,...,4.21,-0.24,0.24,49.3,<NA>,<NA>,<NA>,<NA>,<NA>,travis shaw
1,FF,2021-05-01,94.0,-1.13,5.98,"White, Mitch",543768,669952,NaN,foul,...,1.06,0.23,-0.23,51.3,<NA>,<NA>,<NA>,<NA>,<NA>,travis shaw
2,CU,2021-05-01,82.4,-1.12,5.96,"White, Mitch",543768,669952,NaN,called_strike,...,4.47,-0.3,0.3,50.6,<NA>,<NA>,<NA>,<NA>,<NA>,travis shaw
3,FF,2021-05-01,95.5,-1.0,5.94,"White, Mitch",541645,669952,single,hit_into_play,...,1.06,0.48,0.48,51.4,<NA>,<NA>,<NA>,<NA>,<NA>,avisaíl garcía
4,SL,2021-05-01,88.8,-1.14,5.8,"White, Mitch",541645,669952,NaN,blocked_ball,...,2.36,-0.11,-0.11,48.7,<NA>,<NA>,<NA>,<NA>,<NA>,avisaíl garcía


In [14]:
# select features
features = ["player_name", "batter_name", "pitch_type", "description", "plate_x", "plate_z", "sz_top", "sz_bot", "release_speed", "release_spin_rate", 
            "pitch_type", "balls", "strikes", "pfx_x", "pfx_z", "stand", "p_throws",
            "release_extension"]
df = all_data[features + ["launch_speed", "launch_angle"]]

In [15]:
all_data.shape

(2136337, 119)

In [16]:
# drop na
df = df.dropna(subset=features)

In [17]:
df.head()

,player_name,batter_name,pitch_type,description,plate_x,plate_z,sz_top,sz_bot,release_speed,release_spin_rate,pitch_type,balls,strikes,pfx_x,pfx_z,stand,p_throws,release_extension,launch_speed,launch_angle
0,"White, Mitch",travis shaw,CU,hit_into_play,-0.07,2.28,3.55,1.63,83.0,2455,CU,0,2,0.24,-0.93,L,R,6.0,105.5,14
1,"White, Mitch",travis shaw,FF,foul,-0.08,3.35,3.55,1.63,94.0,2422,FF,0,1,-0.23,1.51,L,R,6.2,100.3,1
2,"White, Mitch",travis shaw,CU,called_strike,0.72,1.84,3.55,1.63,82.4,2551,CU,0,0,0.3,-1.09,L,R,6.1,<NA>,<NA>
3,"White, Mitch",avisaíl garcía,FF,hit_into_play,-0.2,3.46,3.55,1.63,95.5,2498,FF,1,2,-0.48,1.43,R,R,6.2,93.2,4
4,"White, Mitch",avisaíl garcía,SL,blocked_ball,2.53,-0.66,3.3,1.73,88.8,2403,SL,0,2,0.11,0.52,R,R,6.3,<NA>,<NA>


In [18]:
print(f"{df.shape[0]} pitches tracked were thrown between 2021 - 2023 regular season")

2126091 pitches tracked were thrown between 2021 - 2023 regular season


In [19]:
# determine whether a pitch is a ball
def is_ball(pitch):
    """
    Determine whether the given pitch is a ball or not.
    """

    # get data
    plate_x = pitch["plate_x"]
    plate_z = pitch["plate_z"]
    sz_top = pitch["sz_top"]
    sz_bot = pitch["sz_bot"]
    
    # ball radius
    ball = 0.02
    # check x axis
    in_x = abs(plate_x) <= 0.83 + ball
    # check z axis
    in_z = sz_bot - ball <= plate_z <= sz_top + ball

    return not (in_x and in_z)

In [20]:
# filter dataframe with only pitch is a ball
# df = df.dropna()
df = df[df.apply(is_ball, axis=1) == True]

print(f"{df.shape[0]} balls tracked were thrown between 2021 - 2023 regular season")

1133004 balls tracked were thrown between 2021 - 2023 regular season


In [21]:
df.head()

,player_name,batter_name,pitch_type,description,plate_x,plate_z,sz_top,sz_bot,release_speed,release_spin_rate,pitch_type,balls,strikes,pfx_x,pfx_z,stand,p_throws,release_extension,launch_speed,launch_angle
4,"White, Mitch",avisaíl garcía,SL,blocked_ball,2.53,-0.66,3.3,1.73,88.8,2403,SL,0,2,0.11,0.52,R,R,6.3,<NA>,<NA>
9,"White, Mitch",keston hiura,SL,swinging_strike,1.1,1.81,3.37,1.53,88.2,2388,SL,0,1,0.29,0.66,R,R,6.3,<NA>,<NA>
12,"White, Mitch",kolten wong,FF,foul,-0.6,3.3,3.18,1.47,94.1,2385,FF,1,1,-0.33,1.46,L,R,6.2,75.5,31
13,"White, Mitch",kolten wong,CU,ball,-0.41,3.85,3.11,1.42,81.4,2440,CU,0,1,0.42,-1.23,L,R,6.1,<NA>,<NA>
15,"Vesia, Alex",mario feliciano,FF,ball,-1.2,1.2,3.24,1.57,94.2,2446,FF,3,2,0.65,1.81,R,L,6.7,<NA>,<NA>


In [23]:
# determine whether the batter swing or not
def is_swing(description):
    """
    Determine whether the batter swing or take based on given description.
    1 is swing, and 0 is take.
    """

    # swing
    swing_keywords = [
        'swinging_strike', 'swinging_strike_blocked',
        'foul', 'foul_tip', 'foul_bunt',
        'hit_into_play', 'hit_into_play_no_out',
        'hit_into_play_score', 'bunt_foul_tip', 'missed_bunt'
    ]
    # take
    take_keywords = [
        'ball', 'called_strike', 'blocked_ball', 'pitchout', 'hit_by_pitch'
    ]

    # determine result
    if description in swing_keywords:
        return 1
    else:
        return 0

In [24]:
# apply to dataframe
df["swing"] = df["description"].apply(is_swing)

In [25]:
print("number of swing and take")
df["swing"].value_counts()

number of swing and take


swing
0    796182
1    336822
Name: count, dtype: int64

In [26]:
# compute weights
weights = df["swing"].value_counts()[0] / df["swing"].value_counts()[1]

In [27]:
df.head()

,player_name,batter_name,pitch_type,description,plate_x,plate_z,sz_top,sz_bot,release_speed,release_spin_rate,...,balls,strikes,pfx_x,pfx_z,stand,p_throws,release_extension,launch_speed,launch_angle,swing
4,"White, Mitch",avisaíl garcía,SL,blocked_ball,2.53,-0.66,3.3,1.73,88.8,2403,...,0,2,0.11,0.52,R,R,6.3,<NA>,<NA>,0
9,"White, Mitch",keston hiura,SL,swinging_strike,1.1,1.81,3.37,1.53,88.2,2388,...,0,1,0.29,0.66,R,R,6.3,<NA>,<NA>,1
12,"White, Mitch",kolten wong,FF,foul,-0.6,3.3,3.18,1.47,94.1,2385,...,1,1,-0.33,1.46,L,R,6.2,75.5,31,1
13,"White, Mitch",kolten wong,CU,ball,-0.41,3.85,3.11,1.42,81.4,2440,...,0,1,0.42,-1.23,L,R,6.1,<NA>,<NA>,0
15,"Vesia, Alex",mario feliciano,FF,ball,-1.2,1.2,3.24,1.57,94.2,2446,...,3,2,0.65,1.81,R,L,6.7,<NA>,<NA>,0


In [28]:
# normalize plate_z
def normalize_plate_z(pitch):
    """
    Normalize given pitch plate z value.
    """

    # get values
    plate_z = pitch["plate_z"]
    sz_top = pitch["sz_top"]
    sz_bot = pitch["sz_bot"]

    # normalize z
    norm_z = (plate_z - sz_bot) / (sz_top - sz_bot)

    return norm_z

In [29]:
# apply plate_z normalization to dataframe
df["norm_plate_z"] = df.apply(normalize_plate_z, axis=1)

In [30]:
df.head()

,player_name,batter_name,pitch_type,description,plate_x,plate_z,sz_top,sz_bot,release_speed,release_spin_rate,...,strikes,pfx_x,pfx_z,stand,p_throws,release_extension,launch_speed,launch_angle,swing,norm_plate_z
4,"White, Mitch",avisaíl garcía,SL,blocked_ball,2.53,-0.66,3.3,1.73,88.8,2403,...,2,0.11,0.52,R,R,6.3,<NA>,<NA>,0,-1.522293
9,"White, Mitch",keston hiura,SL,swinging_strike,1.1,1.81,3.37,1.53,88.2,2388,...,1,0.29,0.66,R,R,6.3,<NA>,<NA>,1,0.152174
12,"White, Mitch",kolten wong,FF,foul,-0.6,3.3,3.18,1.47,94.1,2385,...,1,-0.33,1.46,L,R,6.2,75.5,31,1,1.070175
13,"White, Mitch",kolten wong,CU,ball,-0.41,3.85,3.11,1.42,81.4,2440,...,1,0.42,-1.23,L,R,6.1,<NA>,<NA>,0,1.437870
15,"Vesia, Alex",mario feliciano,FF,ball,-1.2,1.2,3.24,1.57,94.2,2446,...,2,0.65,1.81,R,L,6.7,<NA>,<NA>,0,-0.221557


In [31]:
# output the cleaned data
df.to_csv("cleaned_data.csv")

In [ ]:
# prepare data function
def prepare_data(df, y="swing"):
    """
    With given dataframe, return X and y if label is equal to 1.
    """
    
    # label pitch_type
    df["label_pitch_type"] = df["pitch_type"].map(PITCH_TYPE)
    df["label_pitch_type"] = df["label_pitch_type"].astype(int)

    # determine it is same or different side
    df["same_side"] = (df["stand"] == df["p_throws"]).astype(int)

    # drop pitchout rows
    df = df[df["description"] != "pitchout"]

    # normalize numerical features
    numerical_features = ["plate_x", "norm_plate_z", "release_speed", "release_spin_rate", 
                          "pfx_x", "pfx_z", "release_extension"]
    scaler = StandardScaler()
    # scaler = MinMaxScaler()
    df[numerical_features] = scaler.fit_transform(df[numerical_features])

    # select data
    features = ["plate_x", "norm_plate_z", "release_speed", "release_spin_rate", "pfx_x", "pfx_z", 
                "balls", "strikes", "same_side", "label_pitch_type",
                "release_extension"]
    X = df[features].values
    y = df[y].values

    return X, y, len(features)

In [34]:
# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim=14):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc2 = nn.Linear(16, 32)
        self.fc3 = nn.Linear(32, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        logits = self.output(x)

        return logits

In [35]:
# set device
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "mps"

In [36]:
def train_model(model, epochs, learning_rate, conv1d=0):
    """
    Train the given model and return training and evaluation result.
    """

    # split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)

    # create tensor
    X_train = X_train.astype(np.float32) # convert data type
    X_test = X_test.astype(np.float32) # convert data type
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

    # create datasets
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    # create DataLoaders
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    # setup training
    pos_weight = torch.tensor(weights, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # create loss list
    train_loss_lst = []
    brier_lst = []
    log_loss_lst = []

    # check what type of model
    if conv1d == 1:
        X_test_tensor = X_test_tensor.unsqueeze(1) # reshape data

    # training model and evaluate on testing set every 10 epoch
    for epoch in range(epochs):
        model.train()
        training_loss = 0

        # update number of batch
        i = 0
        for batch_x, batch_y in train_loader:
            # set optimizer
            optimizer.zero_grad()
            if conv1d == 1:
                batch_x = batch_x.unsqueeze(1)
            outputs = model(batch_x)
            batch_y = batch_y.view(-1, 1)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            training_loss = training_loss + loss.item()
            i = i + 1
            # print training status
            if i % 100 == -1:
                print(f"batch number {i}, training loss: {training_loss}")
        
        # update training loss
        train_loss_lst.append(training_loss / len(train_loader))

        # evaluate on testing set
        # acc_lst = [] # accuracy list
        model.eval()
        with torch.no_grad():
            # for batch_x, batch_y in test_loader:
            #     # get predicted result
            #     logits = swing_take_model(batch_x)
            #     probs = torch.softmax(logits, dim=1)
            #     y_pred = torch.argmax(probs, dim=1).cpu()
            #     y = batch_y.cpu()

            #     # compute accuracy score
            #     acc = accuracy_score(y, y_pred)
            #     acc_lst.append(acc)

            # get predicted value
            logits = model(X_test_tensor)
            y_prob = torch.sigmoid(logits).cpu().numpy().flatten()
            y_test = y_test_tensor.cpu().numpy()
            y_pred = [round(i) for i in y_prob]

            # evaluation score
            brier = brier_score_loss(y_test, y_prob)
            brier_lst.append(brier)
            log = log_loss(y_test, y_prob)
            log_loss_lst.append(log)
            acc = accuracy_score(y_test, y_pred)
            # recall = recall_score(y, y_pred)
            # precision = precision_score(y, y_pred)

            # print out result
            print(f"epoch {epoch}, brier score: {brier}, log loss: {log}")
            # print(f"accuracy: {acc}, recall: {recall}, precision: {precision}")
    
    return train_loss_lst, brier_lst, log_loss_lst

In [37]:
# prepare data
X, y, input_dim = prepare_data(df)

# paramters
learning_rate = 1e-3
epochs = 5

# create MLP model
mlp_model = MLP(input_dim=input_dim).to(device)

# train model
train_loss, brier, log = train_model(mlp_model, epochs, learning_rate)

/var/folders/r1/fdb0tbq57qj3ww3lz_lh0xf00000gn/T/ipykernel_14818/856172203.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[numerical_features] = scaler.fit_transform(df[numerical_features])


epoch 0, brier score: 0.17373660010251588, log loss: 0.5124416439159017
epoch 1, brier score: 0.17892912354792864, log loss: 0.5262056172880748
epoch 2, brier score: 0.1572211795004508, log loss: 0.4691958379464457
epoch 3, brier score: 0.16234834748498886, log loss: 0.48246503370895166
epoch 4, brier score: 0.16879290149470674, log loss: 0.5001281833650454


In [38]:
# build model with all data
# convert type
X = X.astype(np.float32)
y = y.astype(np.float32)

# train model with all data
X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y, dtype=torch.float32).to(device)

# create a dataset
dataset = TensorDataset(X_tensor, y_tensor)

# create a DataLoader with batch size 256
loader = DataLoader(dataset, batch_size=256, shuffle=True)

# paramters
learning_rate = 1e-3
epochs = 5

# create model
model = MLP(input_dim=input_dim).to(device)

# setup training
pos_weight = torch.tensor(weights, dtype=torch.float32)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# train model
train_loss_lst = []
for epoch in range(epochs):
    model.train()
    training_loss = 0

    # update number of batch
    i = 0
    for batch_x, batch_y in loader:
        # set optimizer
        optimizer.zero_grad()
        outputs = model(batch_x)
        batch_y = batch_y.view(-1, 1)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        training_loss = training_loss + loss.item()
        i = i + 1
        # print training status
        if i % 100 == -1:
            print(f"batch number {i}, training loss: {training_loss}")
        
    # update training loss
    train_loss_lst.append(training_loss / len(loader))

    print(f"epoch {epoch} completed.")

epoch 0 completed.
epoch 1 completed.
epoch 2 completed.
epoch 3 completed.
epoch 4 completed.


In [39]:
torch.save(model.state_dict(), "swing_prob_model.pt")

In [40]:
model = MLP(input_dim=input_dim)
model.load_state_dict(torch.load("swing_prob_model.pt"))

<All keys matched successfully>

In [41]:
model = model.to(device)
model.eval()
with torch.no_grad():
    # get predicted value
    logits = model(X_tensor)
    y_prob = torch.sigmoid(logits).cpu().numpy().flatten()
    y_test = y_tensor.cpu().numpy()
    y_pred = [round(i) for i in y_prob]

    # get evaluation metric value
    brier = brier_score_loss(y_test, y_prob)
    log = log_loss(y_test, y_prob)

print(f"brier score {brier}, log loss {log}")

brier score 0.1718709577549243, log loss 0.507238984908404


In [162]:
# compute contact quality for each pitch
def compute_cq(pitch):
    """
    With given pitch, calculate its contact quality.
    """

    # check swing or not
    if pitch["swing"] == 0:
        cq = 0
        return cq

    # get description
    description = pitch["description"]

    if description in ["swinging_strike", "swinging_strike_blocked", "foul", "foul_tip", "foul_bunt", "missed_bunt",
                       "bunt_foul_pitch"]: # swing and miss or foul
        cq = 0
    else:
        # get values
        ev = pitch["launch_speed"] # exit velocity
        la = pitch["launch_angle"] # launch angle
        try:
            # compute scores
            ev_score = min(1, max(0, (ev - 70) / 28))
            la_score = max(0, 1 - abs(la - 20) / 20)

            # compute cq
            cq = ev_score * la_score
        except TypeError:
            cq = None

    return cq